# LeetCode 3375 — Minimum Operations to Make Array Values Equal to K bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-array-values-equal-to-k/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Hash Table (bảng băm / `set`)

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; mọi đáp án được kiểm chứng bằng code, kể cả một bộ **vét cạn BFS** (thử mọi giá trị `h` hợp lệ ở mỗi bước) để chứng tỏ đáp án là **nhỏ nhất thật sự**. Ví dụ tự dựng thêm được ghi rõ "tự dựng" — nếu cần khớp từng chữ thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một số nguyên `h` gọi là **hợp lệ** nếu mọi phần tử **lớn hơn `h` đều bằng nhau**; mỗi phép toán chọn `h` hợp lệ rồi **hạ mọi phần tử `> h` xuống `h`**. Tìm số phép toán **ít nhất** để cả mảng **bằng `k`** (hoặc `-1` nếu không thể).
2. Nắm các công cụ Python: `set` (tập hợp, loại trùng), `len`, `min`, `sorted`, tập hợp dựng bằng biểu thức `{x for x in nums if x > k}`, và việc `True`/`False` cộng trừ được như `1`/`0`.
3. Đi từ cách **mô phỏng đúng đề** (luôn chọn `h` = giá trị lớn thứ nhì) → **nhận ra chỉ cần đếm số giá trị khác nhau lớn hơn `k`** → **một lượt duyệt với `set` và `min`** → **mảng đếm theo giá trị**.
4. Hiểu mấu chốt: mỗi phép toán **gộp tối đa một giá trị phân biệt** (lớp lớn nhất) xuống `h`, và giá trị chỉ **giảm** nên không thể đưa phần tử nhỏ hơn `k` lên `k`. Đáp án = số giá trị phân biệt **lớn hơn `k`**, hoặc `-1` nếu có phần tử `< k`.
5. Tránh các lỗi kinh điển: quên trả `-1` khi có phần tử `< k`, đếm cả giá trị bằng `k`, trừ `1` vô điều kiện, đếm **phần tử** thay vì **giá trị phân biệt**, và dùng `<=` thay `<` khi kiểm tra "bất khả thi".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và một số nguyên `k`.

Một số nguyên `h` gọi là **hợp lệ** nếu **tất cả** các phần tử của mảng **lớn hơn `h` (nghiêm ngặt)** đều **bằng nhau**. Ví dụ `nums = [10, 8, 10, 8]`: `h = 9` hợp lệ (chỉ có các số `10`, giống nhau, lớn hơn `9`), còn `h = 5` **không** hợp lệ (có cả `10` và `8` lớn hơn `5`, chúng khác nhau).

Một **phép toán** gồm:

1. chọn một số nguyên `h` **hợp lệ với mảng hiện tại**;
2. với mọi chỉ số `i` mà `nums[i] > h`, gán `nums[i] = h`.

Trả về **số phép toán ít nhất** để **mọi phần tử bằng `k`**. Nếu không thể, trả `-1`.

**Ví dụ 1**

- Input: `nums = [5, 2, 5, 4, 5], k = 2`
- Output: `2`
- Giải thích: chọn `h = 4` (mảng thành `[4, 2, 4, 4, 4]`), rồi chọn `h = 2` (thành `[2, 2, 2, 2, 2]`).

**Ví dụ 2**

- Input: `nums = [2, 1, 2], k = 2`
- Output: `-1`
- Giải thích: có phần tử `1 < 2`, mà phép toán chỉ **hạ** giá trị xuống nên không bao giờ nâng `1` lên `2` được.

**Ví dụ 3**

- Input: `nums = [9, 7, 5, 3], k = 1`
- Output: `4`
- Giải thích: lần lượt `h = 7, 5, 3, 1`.

**Ràng buộc**

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 100`
- `1 <= k <= 100`

In [1]:
from typing import List

examples = [
    (([5, 2, 5, 4, 5], 2), 2),
    (([2, 1, 2], 2), -1),
    (([9, 7, 5, 3], 1), 4),
]

def min_ops_quick(nums, k):
    """Bản nhanh để chạy thử ví dụ: đếm giá trị phân biệt > k (giải thích ở các mục sau)."""
    if min(nums) < k:
        return -1
    return len({x for x in nums if x > k})

for (nums, k), expected in examples:
    print(f"nums={nums}, k={k} -> {min_ops_quick(nums, k)} (kỳ vọng {expected})")
    assert min_ops_quick(nums, k) == expected

nums=[5, 2, 5, 4, 5], k=2 -> 2 (kỳ vọng 2)
nums=[2, 1, 2], k=2 -> -1 (kỳ vọng -1)
nums=[9, 7, 5, 3], k=1 -> 4 (kỳ vọng 4)


## 3. Kiến thức nền

### 3.1. Đọc kỹ định nghĩa `h` hợp lệ

"Mọi phần tử **lớn hơn `h`** đều **giống nhau**." Ta kiểm tra bằng code: lấy các phần tử `> h` rồi xem tập các giá trị đó có **nhiều nhất 1 phần tử** không. Chú ý hai chi tiết: (a) so sánh **nghiêm ngặt** `>`; (b) nếu **không có** phần tử nào `> h` thì điều kiện đúng "hiển nhiên" (không có gì để khác nhau) — ví dụ `h = 10` với `[10, 8, 10, 8]` cũng hợp lệ (nhưng không làm thay đổi gì).

In [2]:
def is_valid(h, nums):
    above = {x for x in nums if x > h}     # tập các giá trị lớn hơn h
    return len(above) <= 1                 # 0 hoặc 1 giá trị

nums = [10, 8, 10, 8]
for h in (5, 7, 8, 9, 10, 11):
    above = sorted({x for x in nums if x > h})
    print(f"h={h:>2}: các giá trị > h = {above} -> hợp lệ? {is_valid(h, nums)}")

h= 5: các giá trị > h = [8, 10] -> hợp lệ? False
h= 7: các giá trị > h = [8, 10] -> hợp lệ? False
h= 8: các giá trị > h = [10] -> hợp lệ? True
h= 9: các giá trị > h = [10] -> hợp lệ? True
h=10: các giá trị > h = [] -> hợp lệ? True
h=11: các giá trị > h = [] -> hợp lệ? True


### 3.2. `set` — tập hợp, tự loại trùng

`set` là tập hợp các giá trị **không trùng lặp**. Tạo từ danh sách bằng `set(nums)`, thêm bằng `.add(x)`, đếm số phần tử **phân biệt** bằng `len(...)`. Đây chính là công cụ để đếm "bao nhiêu giá trị **khác nhau**".

In [3]:
nums = [5, 2, 5, 4, 5]
s = set(nums)
print(s)                 # {2, 4, 5}  (thứ tự không quan trọng)
print(len(nums), len(s)) # 5 phần tử, nhưng 3 giá trị phân biệt

t = set()
for x in nums:
    t.add(x)             # thêm giá trị đã có thì không đổi gì
print(t == s)            # True
print(4 in s, 7 in s)    # True False

{2, 4, 5}
5 3
True
True False


### 3.3. Tập hợp dựng bằng điều kiện: `{x for x in nums if x > k}`

Đây là "set comprehension": duyệt `nums`, chỉ lấy `x` thoả điều kiện, tự loại trùng. Ví dụ `{x for x in nums if x > 2}` với `[5, 2, 5, 4, 5]` cho `{5, 4}`.

In [4]:
nums = [5, 2, 5, 4, 5]
k = 2
above_k = {x for x in nums if x > k}
print(above_k, len(above_k))                    # {4, 5} 2

# Tương đương vòng lặp for thông thường
res = set()
for x in nums:
    if x > k:
        res.add(x)
print(res == above_k)                           # True

{4, 5} 2
True


### 3.4. `min`, `sorted` và `True/False` như số

- `min(nums)` cho phần tử nhỏ nhất; `sorted(set(nums))` cho các giá trị phân biệt theo thứ tự tăng.
- Trong Python `True == 1` và `False == 0`, nên `len(s) - (mi == k)` trừ `1` khi `mi == k` đúng và trừ `0` khi sai. Ta thường bọc bằng `int(...)` cho rõ nghĩa: `len(s) - int(mi == k)`.

In [5]:
nums = [9, 7, 5, 3]
print(min(nums))                    # 3
print(sorted(set(nums)))            # [3, 5, 7, 9]
print(sorted(set(nums), reverse=True))   # [9, 7, 5, 3]
print(True + True, int(3 == 3), int(3 == 4))   # 2 1 0

3
[3, 5, 7, 9]
[9, 7, 5, 3]
2 1 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `nums = [5, 2, 5, 4, 5]`, `k = 2`. Mỗi phép toán hạ **toàn bộ lớp cao nhất** xuống `h`:

| bước | mảng | giá trị phân biệt > 2 | chọn `h` | hợp lệ vì |
|---|---|---|---|---|
| 0 | `[5, 2, 5, 4, 5]` | `{5, 4}` (2 giá trị) | `4` | phần tử `> 4` chỉ có `5`, `5`, `5` |
| 1 | `[4, 2, 4, 4, 4]` | `{4}` (1 giá trị) | `2` | phần tử `> 2` chỉ là các `4` |
| 2 | `[2, 2, 2, 2, 2]` | `{}` (0 giá trị) | — | xong |

Số giá trị phân biệt `> k` giảm từ `2` xuống `0` — **đúng 2 phép toán**. Ba nhận xét:

1. Mỗi phép toán chỉ "ăn" được **lớp lớn nhất** (vì `h` hợp lệ đòi hỏi mọi thứ `> h` giống nhau, tức đúng một giá trị — chính là giá trị lớn nhất).
2. Để chạm tới `k`, ta phải "đi qua" mọi giá trị phân biệt lớn hơn `k` — mỗi giá trị cần một phép toán riêng.
3. Nếu có số `< k` thì **vô vọng**: phép toán chỉ làm số **nhỏ đi**.

In [6]:
def show_greedy(nums, k):
    nums = list(nums)
    print("ban đầu:", nums, "k =", k)
    step = 0
    while max(nums) > k:
        m = max(nums)
        below = [x for x in nums if x < m]
        h = max(below) if below else k           # lớp kế tiếp, hoặc k nếu chỉ còn một lớp
        h = max(h, k)
        nums = [h if x > h else x for x in nums]
        step += 1
        print(f"  bước {step}: chọn h={h} -> {nums}")
    return step

show_greedy([5, 2, 5, 4, 5], 2)
show_greedy([9, 7, 5, 3], 1)

ban đầu: [5, 2, 5, 4, 5] k = 2
  bước 1: chọn h=4 -> [4, 2, 4, 4, 4]
  bước 2: chọn h=2 -> [2, 2, 2, 2, 2]
ban đầu: [9, 7, 5, 3] k = 1
  bước 1: chọn h=7 -> [7, 7, 5, 3]
  bước 2: chọn h=5 -> [5, 5, 5, 3]
  bước 3: chọn h=3 -> [3, 3, 3, 3]
  bước 4: chọn h=1 -> [1, 1, 1, 1]


4

## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng đề (luôn chọn `h` là giá trị lớn thứ nhì)

**Ý tưởng.** Làm đúng như đề: lặp "chọn `h` hợp lệ → hạ mọi phần tử `> h`". Chọn `h` thế nào? Phần tử lớn nhất hiện tại `m` buộc phải bị hạ; để **hạ ít nhất có thể trong mỗi bước** ta chọn `h` = **giá trị lớn nhất nhỏ hơn `m`** (nếu có), nếu không còn giá trị nào nhỏ hơn `m` thì chọn `h = k`. Trước khi bắt đầu, nếu có phần tử `< k` thì trả `-1`.

Đếm số vòng lặp đến khi `max(nums) == k`.

In [7]:
def min_ops_simulate(nums: List[int], k: int) -> int:
    nums = list(nums)                    # copy để không sửa input
    if min(nums) < k:
        return -1
    ops = 0
    while max(nums) > k:
        m = max(nums)
        below = [x for x in nums if x < m]
        h = max(below) if below else k   # giá trị lớn nhì (>= k vì mọi x >= k), hoặc k
        nums = [h if x > h else x for x in nums]
        ops += 1
    return ops

for (nums, k), expected in examples:
    print(min_ops_simulate(nums, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
4 (kỳ vọng 4 )


**Độ phức tạp (Cách 1).** Số phép toán tối đa bằng số giá trị phân biệt (`<= 100`); mỗi phép toán duyệt mảng `O(n)` ⇒ **Time `O(n * d)`** với `d` là số giá trị phân biệt (`<= n`), tức `O(n^2)` tệ nhất; **Space `O(n)`** cho bản sao.

**Nút thắt.** Ta không cần **thực sự biến đổi mảng** để biết có bao nhiêu phép toán. Số phép toán chỉ phụ thuộc vào **số giá trị phân biệt lớn hơn `k`**; nhìn ra điều này thì một lượt duyệt là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

**Đếm số giá trị phân biệt `> k`.** Gọi `D = { x trong nums : x > k }` (một tập, trùng bị loại). Khi `nums` không có phần tử `< k`:

> **đáp án = `|D|`** (số phần tử của `D`).

Vì sao `|D|` là **đủ**: mỗi phép toán chọn `h` = giá trị lớn nhì (hoặc `k`) gộp lớp lớn nhất vào lớp kế tiếp ⇒ `|D|` giảm đúng `1` (hoặc `D` rỗng thì dừng).

Vì sao `|D|` là **cần**: mỗi phép toán hợp lệ **không thể giảm `|D|` quá `1`** — các giá trị `> h` đều bằng nhau (một giá trị duy nhất `v`), chúng chỉ biến thành `h`; nếu `h > k` mà `h` đã có sẵn thì `|D|` giảm `1`, nếu `h` chưa có thì `|D|` không đổi; nếu `h = k` thì `v` rời khỏi `D`, giảm `1`. Vậy cần **ít nhất `|D|`** phép toán.

Ghi nhớ gọn: **`-1` nếu có `x < k`; ngược lại số giá trị phân biệt `> k`.**

In [8]:
for (nums, k), expected in examples:
    D = {x for x in nums if x > k}
    print(f"nums={nums}, k={k}: D = {sorted(D, reverse=True)}, |D| = {len(D)}, có x<k? {min(nums) < k}, kỳ vọng {expected}")

nums=[5, 2, 5, 4, 5], k=2: D = [5, 4], |D| = 2, có x<k? False, kỳ vọng 2
nums=[2, 1, 2], k=2: D = [], |D| = 0, có x<k? True, kỳ vọng -1
nums=[9, 7, 5, 3], k=1: D = [9, 7, 5, 3], |D| = 4, có x<k? False, kỳ vọng 4


## 7. Các cách tiếp theo

### Cách 2 — Đếm giá trị phân biệt `> k` bằng `set` comprehension

**Giải thích từng đoạn.**

1. `if min(nums) < k: return -1` — có phần tử nhỏ hơn `k` thì bất khả thi.
2. `{x for x in nums if x > k}` — tập các giá trị phân biệt **lớn hơn `k`**.
3. `return len(...)` — mỗi giá trị cần đúng một phép toán.

In [9]:
def min_ops_set(nums: List[int], k: int) -> int:
    if min(nums) < k:
        return -1
    return len({x for x in nums if x > k})

for (nums, k), expected in examples:
    print(min_ops_set(nums, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
4 (kỳ vọng 4 )


In [10]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from typing import List

class SolutionSet:
    def minOperations(self, nums: List[int], k: int) -> int:
        if min(nums) < k:
            return -1
        return len({x for x in nums if x > k})

print(SolutionSet().minOperations([5, 2, 5, 4, 5], 2))   # 2

2


**Độ phức tạp Cách 2:** **Time `O(n)`** (hai lượt: `min` và dựng `set`), **Space `O(n)`** cho tập.

### Cách 3 — Một lượt duyệt: `set` + `min` (lời giải chính thức)

**Giải thích.** Gộp hai việc vào một vòng `for`:

1. Với mỗi `x`: nếu `x < k` thì trả `-1` ngay.
2. Thêm `x` vào `s` (tập mọi giá trị **phân biệt**, kể cả bằng `k`) và cập nhật `mi = min(mi, x)`.
3. Cuối cùng `len(s)` đếm **cả** giá trị `k` nếu có trong mảng; giá trị `k` **không cần phép toán**, nên trừ `1` **khi `mi == k`** (nghĩa là `k` có mặt, vì không có số nào `< k`).
4. Trả `len(s) - int(mi == k)`.

Nếu `mi > k` thì `k` không có trong mảng, và mọi giá trị trong `s` đều `> k` cần xử lý ⇒ không trừ.

In [11]:
def min_ops_onepass(nums: List[int], k: int) -> int:
    s = set()
    mi = float("inf")
    for x in nums:
        if x < k:
            return -1
        mi = min(mi, x)
        s.add(x)
    return len(s) - int(mi == k)

for (nums, k), expected in examples:
    print(min_ops_onepass(nums, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
4 (kỳ vọng 4 )


In [12]:
# Code nộp LeetCode cho Cách 3
from typing import List

class SolutionOnePass:
    def minOperations(self, nums: List[int], k: int) -> int:
        s = set()
        mi = float("inf")
        for x in nums:
            if x < k:
                return -1
            mi = min(mi, x)
            s.add(x)
        return len(s) - int(mi == k)

print(SolutionOnePass().minOperations([9, 7, 5, 3], 1))   # 4

4


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (tập `s`; tối đa 100 giá trị).

### Cách 4 — Mảng đếm theo giá trị (vì `nums[i] <= 100`)

**Giải thích.** Do giá trị chỉ từ `1` đến `100`, ta dùng **mảng boolean** `seen[v]` ("đã thấy giá trị `v`") thay cho `set`. Duyệt `nums` đánh dấu `seen[x] = True` (và kiểm tra `x < k`). Cuối cùng đếm `v` trong `k + 1 .. 100` có `seen[v]`. Đây là "counting array", thường nhanh hơn băm khi miền giá trị nhỏ và biết trước.

In [13]:
def min_ops_count(nums: List[int], k: int) -> int:
    seen = [False] * 101                 # chỉ số 0..100
    for x in nums:
        if x < k:
            return -1
        seen[x] = True
    return sum(1 for v in range(k + 1, 101) if seen[v])

for (nums, k), expected in examples:
    print(min_ops_count(nums, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
-1 (kỳ vọng -1 )
4 (kỳ vọng 4 )


In [14]:
# Code nộp LeetCode cho Cách 4
from typing import List

class SolutionCount:
    def minOperations(self, nums: List[int], k: int) -> int:
        seen = [False] * 101
        for x in nums:
            if x < k:
                return -1
            seen[x] = True
        return sum(1 for v in range(k + 1, 101) if seen[v])

print(SolutionCount().minOperations([2, 1, 2], 2))   # -1

-1


**Độ phức tạp Cách 4:** **Time `O(n + V)`** với `V = 100` là miền giá trị, **Space `O(V)`** (hằng số).

**Tóm lại** ba cách đếm (2, 3, 4) cho cùng đáp án; Cách 1 là bản mô phỏng đúng đề để ta **tin** vào công thức đếm. Ở mục 12 ta còn kiểm chứng bằng **BFS vét cạn**.

## 8. TRACE từng bước

Trace **Cách 3** với `nums = [5, 2, 5, 4, 5]`, `k = 2`. Cột `s` là tập sau khi thêm `x`.

| i | x | x < k? | mi (sau) | s (sau) |
|---|---|---|---|---|
| 0 | 5 | không | 5 | {5} |
| 1 | 2 | không | 2 | {5, 2} |
| 2 | 5 | không | 2 | {5, 2} |
| 3 | 4 | không | 2 | {5, 2, 4} |
| 4 | 5 | không | 2 | {5, 2, 4} |

Kết thúc: `len(s) = 3`, `mi == k` (`2 == 2`) nên trừ `1` ⇒ `3 - 1 = 2`. Đúng.

Với `nums = [9, 7, 5, 3]`, `k = 1`: `s = {9, 7, 5, 3}` (4 giá trị), `mi = 3 != 1` nên **không trừ** ⇒ `4`. Với `[2, 1, 2]`, `k = 2`: tại `i = 1`, `x = 1 < 2` ⇒ trả `-1`.

In [15]:
def trace_min_ops(nums, k):
    s = set()
    mi = float("inf")
    print(f"nums={nums}, k={k}")
    for i, x in enumerate(nums):
        if x < k:
            print(f"i={i} x={x}: x < k -> bất khả thi, trả -1")
            return -1
        mi = min(mi, x)
        s.add(x)
        print(f"i={i} x={x}: x<k? không | mi={mi} | s={sorted(s)}")
    sub = int(mi == k)
    print(f"len(s)={len(s)}, mi == k? {mi == k} -> trừ {sub} => {len(s) - sub}")
    return len(s) - sub

trace_min_ops([5, 2, 5, 4, 5], 2)
trace_min_ops([9, 7, 5, 3], 1)
trace_min_ops([2, 1, 2], 2)

nums=[5, 2, 5, 4, 5], k=2
i=0 x=5: x<k? không | mi=5 | s=[5]
i=1 x=2: x<k? không | mi=2 | s=[2, 5]
i=2 x=5: x<k? không | mi=2 | s=[2, 5]
i=3 x=4: x<k? không | mi=2 | s=[2, 4, 5]
i=4 x=5: x<k? không | mi=2 | s=[2, 4, 5]
len(s)=3, mi == k? True -> trừ 1 => 2
nums=[9, 7, 5, 3], k=1
i=0 x=9: x<k? không | mi=9 | s=[9]
i=1 x=7: x<k? không | mi=7 | s=[7, 9]
i=2 x=5: x<k? không | mi=5 | s=[5, 7, 9]
i=3 x=3: x<k? không | mi=3 | s=[3, 5, 7, 9]
len(s)=4, mi == k? False -> trừ 0 => 4
nums=[2, 1, 2], k=2
i=0 x=2: x<k? không | mi=2 | s=[2]
i=1 x=1: x < k -> bất khả thi, trả -1


-1

In [16]:
# Trace Cách 1 (mô phỏng) với ví dụ 3: cho thấy h được chọn từng bước
def trace_simulate(nums, k):
    nums = list(nums)
    print("ban đầu:", nums, "k =", k)
    ops = 0
    while max(nums) > k:
        m = max(nums)
        below = [x for x in nums if x < m]
        h = max(below) if below else k
        print(f"  max={m}, các giá trị nhỏ hơn max={sorted(set(below))} -> chọn h={h}, hợp lệ? {is_valid(h, nums)}")
        nums = [h if x > h else x for x in nums]
        ops += 1
        print(f"  sau phép toán {ops}: {nums}")
    print("tổng:", ops)
    return ops

trace_simulate([9, 7, 5, 3], 1)

ban đầu: [9, 7, 5, 3] k = 1
  max=9, các giá trị nhỏ hơn max=[3, 5, 7] -> chọn h=7, hợp lệ? True
  sau phép toán 1: [7, 7, 5, 3]
  max=7, các giá trị nhỏ hơn max=[3, 5] -> chọn h=5, hợp lệ? True
  sau phép toán 2: [5, 5, 5, 3]
  max=5, các giá trị nhỏ hơn max=[3] -> chọn h=3, hợp lệ? True
  sau phép toán 3: [3, 3, 3, 3]
  max=3, các giá trị nhỏ hơn max=[] -> chọn h=1, hợp lệ? True
  sau phép toán 4: [1, 1, 1, 1]
tổng: 4


4

## 9. Code nộp LeetCode

Ta nộp **Cách 3** (một lượt, đúng với lời giải chính thức). Nếu muốn ngắn hơn, Cách 2 cũng được.

In [17]:
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        s = set()                       # các giá trị phân biệt
        mi = float("inf")               # phần tử nhỏ nhất
        for x in nums:
            if x < k:
                return -1               # không thể nâng x lên k
            mi = min(mi, x)
            s.add(x)
        return len(s) - int(mi == k)    # k không cần phép toán nếu đã có trong mảng


for (nums, k), expected in examples:
    got = Solution().minOperations(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 5, 4, 5] 2 -> 2 OK
[2, 1, 2] 2 -> -1 OK
[9, 7, 5, 3] 1 -> 4 OK


## 10. Vì sao thuật toán đúng

**Trường hợp bất khả thi.** Mỗi phép toán chỉ gán `nums[i] = h` khi `nums[i] > h`, tức giá trị **không bao giờ tăng**. Vậy nếu có `x < k` thì `x` mãi `<= x < k`, không bao giờ bằng `k` ⇒ `-1`. Ngược lại nếu mọi `x >= k` thì luôn làm được (xem bên dưới).

**Cận dưới (cần ít nhất `|D|` phép toán).** `D` = tập giá trị phân biệt `> k`. Với `h` hợp lệ, các phần tử `> h` đều bằng **một** giá trị `v` (nếu có), và chỉ những phần tử này bị đổi thành `h`. Có hai khả năng:

- `h >= k`: tập mới `D'` bằng `D` bỏ `v`, rồi thêm `h` nếu `h > k`. Vậy `|D'| >= |D| - 1` (giảm tối đa `1`).
- `h < k`: các phần tử bị hạ xuống dưới `k` và **không bao giờ quay lại** (giá trị không tăng), nên đường đi này không dẫn tới đích.

Ta cần `|D| = 0`, mà mỗi phép toán giảm `|D|` tối đa `1`, nên cần **ít nhất `|D|`** phép toán.

**Cận trên (làm được trong `|D|` phép toán).** Lặp: nếu `D` còn phần tử, gọi `m = max(D)`, chọn `h = ` giá trị lớn nhất nhỏ hơn `m` (`>= k` vì mọi `x >= k`) — hợp lệ vì chỉ có `m` lớn hơn `h`. Mỗi lần `|D|` giảm đúng 1 ⇒ sau `|D|` phép toán thì `D` rỗng, tức mọi phần tử `<= k`, kết hợp `>= k` ⇒ bằng `k`.

**Công thức của code.** `len(s) - int(mi == k)`: `s` gồm `D` và (nếu `mi == k`) thêm giá trị `k`. Vậy `len(s) - int(mi == k) = |D|`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 100` là miền giá trị.

- **Cách 1 (mô phỏng):** Time `O(n * d)` ≤ `O(n^2)`, Space `O(n)`.
- **Cách 2 (set comprehension):** Time `O(n)`, Space `O(n)`.
- **Cách 3 (một lượt `set` + `min`):** Time `O(n)`, Space `O(n)`.
- **Cách 4 (mảng đếm):** Time `O(n + V)`, Space `O(V)`.

Với `n <= 100` mọi cách đều rất nhanh. Về mặt lý thuyết, mọi cách `O(n)` đều **tối ưu thời gian**, vì phải nhìn mỗi phần tử ít nhất một lần.

## 12. Bộ test

### 12.1. Ví dụ của đề và các trường hợp biên

In [18]:
impls = {
    "simulate": min_ops_simulate,
    "set":      lambda a, k: SolutionSet().minOperations(a, k),
    "onepass":  lambda a, k: Solution().minOperations(a, k),
    "count":    lambda a, k: SolutionCount().minOperations(a, k),
}

cases = [
    ([5, 2, 5, 4, 5], 2, 2),      # ví dụ 1
    ([2, 1, 2], 2, -1),           # ví dụ 2
    ([9, 7, 5, 3], 1, 4),         # ví dụ 3
    ([1], 1, 0),                  # tự dựng: đã bằng k
    ([5], 1, 1),                  # tự dựng: một phần tử lớn hơn k
    ([3, 3, 3], 3, 0),            # tự dựng: tất cả đã bằng k
    ([5, 5, 5], 2, 1),            # tự dựng: nhiều phần tử nhưng chỉ 1 giá trị phân biệt
    ([1, 2, 3], 2, -1),           # tự dựng: có phần tử 1 < 2
    ([4, 6], 4, 1),               # tự dựng: [6, 4] -> h = 4
    ([100] * 100, 1, 1),          # tự dựng: n = 100, một giá trị
    (list(range(1, 101)), 1, 99), # tự dựng: 100 giá trị phân biệt, k = 1 có sẵn
    (list(range(2, 101)), 1, 99), # tự dựng: k = 1 không có trong mảng, 99 giá trị
    (list(range(1, 101)), 100, -1),   # tự dựng: có phần tử < 100
    ([7, 7, 5, 5], 5, 1),         # tự dựng
]

for nums, k, exp in cases:
    for name, f in impls.items():
        got = f(nums, k)
        assert got == exp, (name, nums[:8], k, got, exp)
print("12.1 OK:", len(cases), "test x", len(impls), "cách")

12.1 OK: 14 test x 4 cách


### 12.2. Phản ví dụ cho các cách làm sai

Ta viết các bản sai có chủ đích rồi tìm input làm lộ lỗi.

In [19]:
def wrong_no_neg1(nums, k):
    """Sai: quên trường hợp có phần tử < k."""
    return len({x for x in nums if x > k})

def wrong_count_all(nums, k):
    """Sai: đếm cả giá trị k (len(set(nums)))."""
    if min(nums) < k:
        return -1
    return len(set(nums))

def wrong_always_minus1(nums, k):
    """Sai: luôn trừ 1."""
    if min(nums) < k:
        return -1
    return len(set(nums)) - 1

def wrong_count_elements(nums, k):
    """Sai: đếm phần tử > k thay vì giá trị phân biệt."""
    if min(nums) < k:
        return -1
    return sum(1 for x in nums if x > k)

def wrong_le(nums, k):
    """Sai: x <= k thì trả -1 (x == k vẫn hợp lệ)."""
    if min(nums) <= k:
        return -1
    return len({x for x in nums if x > k})

print("no_neg1       :", wrong_no_neg1([2, 1, 2], 2), "(đúng: -1)")
print("count_all     :", wrong_count_all([5, 2, 5, 4, 5], 2), "(đúng: 2)")
print("always_minus1 :", wrong_always_minus1([9, 7, 5, 3], 1), "(đúng: 4)")
print("count_elements:", wrong_count_elements([5, 5, 5], 2), "(đúng: 1)")
print("le            :", wrong_le([3, 3], 3), "(đúng: 0)")
assert wrong_no_neg1([2, 1, 2], 2) == 0
assert wrong_count_all([5, 2, 5, 4, 5], 2) == 3
assert wrong_always_minus1([9, 7, 5, 3], 1) == 3
assert wrong_count_elements([5, 5, 5], 2) == 3
assert wrong_le([3, 3], 3) == -1
print("12.2 OK: năm bản sai đều bị lộ")

no_neg1       : 0 (đúng: -1)
count_all     : 3 (đúng: 2)
always_minus1 : 3 (đúng: 4)
count_elements: 3 (đúng: 1)
le            : -1 (đúng: 0)
12.2 OK: năm bản sai đều bị lộ


### 12.3. Vét cạn BFS (oracle) và đối chiếu ngẫu nhiên (seed)

Để chắc rằng công thức "đếm giá trị phân biệt" là **nhỏ nhất thật sự**, ta viết **BFS** trên không gian trạng thái: từ mảng hiện tại, thử **mọi** `h` từ `0` đến `max(nums)` (mọi lựa chọn, kể cả `h` kém tối ưu hoặc đưa giá trị xuống dưới `k`), chỉ giữ `h` hợp lệ; tìm số bước ngắn nhất tới mảng toàn `k`. Dùng cho mảng nhỏ (độ dài `<= 6`, giá trị `<= 7`).

In [20]:
from collections import deque
import random

def bfs_oracle(nums, k):
    start = tuple(nums)
    goal = tuple([k] * len(nums))
    if start == goal:
        return 0
    seen = {start}
    dq = deque([(start, 0)])
    while dq:
        state, d = dq.popleft()
        top = max(state)
        for h in range(0, top):                       # h >= top không thay đổi gì
            if len({x for x in state if x > h}) > 1:  # h không hợp lệ
                continue
            nxt = tuple(h if x > h else x for x in state)
            if nxt == goal:
                return d + 1
            if nxt not in seen:
                seen.add(nxt)
                dq.append((nxt, d + 1))
    return -1

assert bfs_oracle([5, 2, 5, 4, 5], 2) == 2
assert bfs_oracle([2, 1, 2], 2) == -1
assert bfs_oracle([9, 7, 5, 3], 1) == 4
print("BFS khớp 3 ví dụ của đề")

random.seed(3375)
total = 0
for _ in range(700):
    n = random.randint(1, 6)
    nums = [random.randint(1, 7) for _ in range(n)]
    k = random.randint(1, 7)
    exp = bfs_oracle(nums, k)
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k, exp, f(nums, k))
    total += 1
print("12.3a OK:", total, "input nhỏ, 4 cách khớp BFS")

# Input lớn đúng ràng buộc: 4 cách khớp nhau
for _ in range(300):
    n = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    k = random.randint(1, 100)
    results = {name: f(nums, k) for name, f in impls.items()}
    assert len(set(results.values())) == 1, (nums, k, results)
print("12.3b OK: 300 input lớn, 4 cách khớp nhau")

BFS khớp 3 ví dụ của đề
12.3a OK: 700 input nhỏ, 4 cách khớp BFS
12.3b OK: 300 input lớn, 4 cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp `-1` (có phần tử `< k`)

Chỉ đếm giá trị phân biệt `> k` mà bỏ qua việc kiểm tra `x < k` sẽ trả sai ở `[2, 1, 2], k = 2` (đáng lẽ `-1`, nhưng code trả `0`).

In [21]:
# SAI
def sai_1(nums, k):
    return len({x for x in nums if x > k})

print(sai_1([2, 1, 2], 2))        # 0 (sai: không thể làm cho 1 thành 2)

# ĐÚNG
print(Solution().minOperations([2, 1, 2], 2))     # -1

0
-1


### Lỗi 2 — Đếm cả giá trị bằng `k`

`len(set(nums))` đếm luôn giá trị `k` — nhưng phần tử đã bằng `k` **không cần** phép toán. Với `[5, 2, 5, 4, 5]`, `k = 2` ta có `set = {5, 4, 2}` (3 giá trị) mà đáp án là `2`.

In [22]:
def sai_2(nums, k):
    if min(nums) < k:
        return -1
    return len(set(nums))          # <-- sai: đếm cả k

print(sai_2([5, 2, 5, 4, 5], 2))   # 3 (sai; đúng là 2)
print(Solution().minOperations([5, 2, 5, 4, 5], 2))   # 2

3
2


### Lỗi 3 — Trừ `1` vô điều kiện

Trừ `1` chỉ đúng khi `k` **có mặt** trong mảng (`mi == k`). Nếu `k` không có (ví dụ `[9, 7, 5, 3]`, `k = 1`), tất cả `4` giá trị đều cần phép toán.

In [23]:
def sai_3(nums, k):
    if min(nums) < k:
        return -1
    return len(set(nums)) - 1      # <-- sai: có thể k không nằm trong nums

print(sai_3([9, 7, 5, 3], 1))      # 3 (sai; đúng là 4)
print(Solution().minOperations([9, 7, 5, 3], 1))      # 4

3
4


### Lỗi 4 — Đếm **phần tử** thay vì giá trị **phân biệt**

Mỗi phép toán hạ **cả lớp** các phần tử bằng nhau cùng lúc, nên chỉ cần **một** phép toán cho mọi bản sao của một giá trị. `[5, 5, 5]`, `k = 2` cần `1` phép toán (`h = 2`), không phải `3`.

In [24]:
def sai_4(nums, k):
    if min(nums) < k:
        return -1
    return sum(1 for x in nums if x > k)      # <-- sai: đếm phần tử

print(sai_4([5, 5, 5], 2))        # 3 (sai; đúng là 1)
print(Solution().minOperations([5, 5, 5], 2))   # 1

3
1


### Lỗi 5 — Dùng `<=` khi kiểm tra bất khả thi

Phần tử **bằng `k`** là hoàn toàn ổn. Chỉ phần tử **`< k`** mới làm bài vô nghiệm. Viết `if x <= k: return -1` làm `[3, 3]`, `k = 3` (đáp án `0`) thành `-1`.

In [25]:
def sai_5(nums, k):
    s = set()
    mi = float("inf")
    for x in nums:
        if x <= k:                 # <-- sai: phải là x < k
            return -1
        mi = min(mi, x)
        s.add(x)
    return len(s) - int(mi == k)

print(sai_5([3, 3], 3))            # -1 (sai; đúng là 0)
print(Solution().minOperations([3, 3], 3))      # 0

-1
0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy đề **phức tạp về luật chơi** (hợp lệ, phép toán) mà ràng buộc nhỏ ⇒ hãy **mô phỏng bằng tay một ví dụ** rồi tìm đại lượng **thay đổi đều đặn** (ở đây: số giá trị phân biệt `> k`).
- "Số phép toán tối thiểu", mỗi phép toán "gộp lớp lớn nhất xuống" ⇒ thường là **đếm số lớp (giá trị phân biệt)**.
- Phép toán **chỉ giảm** (không tăng) ⇒ có ngưỡng bất khả thi: phần tử nhỏ hơn đích ⇒ `-1`.
- **Đếm giá trị phân biệt** ⇒ `len(set(...))`; kết hợp điều kiện lọc bằng set comprehension.
- Đầu bài có miền giá trị nhỏ ⇒ có thể thay `set` bằng **mảng đếm**.
- Khi nghi ngờ công thức: viết **BFS vét cạn** trên input nhỏ (mục 12.3) — cách mạnh nhất để biết "min" thật sự đúng.
- Cẩn thận hai biên: giá trị **bằng `k`** (không tính) và giá trị **nhỏ hơn `k`** (bất khả thi).

## 15. Biến thể của bài

- **Làm cho mọi phần tử bằng nhau (giá trị bất kỳ):** đáp án là `len(set(nums)) - 1` (bài tập 5).
- **Trả về dãy các `h` đã chọn:** dùng tập giá trị phân biệt `> k` sắp giảm dần, thêm `k` ở cuối (bài tập 4).
- **Làm cho mọi phần tử bằng `k` bằng phép tăng/giảm khác:** các luật khác (ví dụ chọn `h` bất kỳ ở mọi lúc) đổi hoàn toàn công thức; luôn kiểm bằng BFS trước.
- **Mảng rất lớn (`n = 10^6`):** vẫn `O(n)` với `set`; nếu giá trị bị chặn nhỏ thì dùng mảng đếm.
- **Đếm số `k` khả thi** (mọi `k` từ `1` đến `min(nums)`): đáp án với `k < min` là `len(set(nums))`, với `k = min` là `len(set(nums)) - 1`.
- **Phép toán nâng lên** (chọn `h` nhỏ hơn mọi phần tử rồi nâng lên): bài đối xứng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | Lặp hạ giá trị lớn nhất xuống giá trị lớn nhì | `O(n * d)` | `O(n)` | Hiểu luật; kiểm chứng công thức |
| 2. Set comprehension | `-1` nếu `min < k`, ngược lại `len({x > k})` | `O(n)` | `O(n)` | Ngắn gọn, dễ đọc |
| 3. Một lượt `set` + `min` | `len(s) - int(mi == k)` | `O(n)` | `O(n)` | **Cách nộp chính** (lời giải chính thức) |
| 4. Mảng đếm | `seen[v]`, đếm `v > k` | `O(n + V)` | `O(V)` | Khi miền giá trị nhỏ, cố định |

Mọi cách cùng đáp án (đối chiếu với BFS ở mục 12.3).

## 17. 5 bài tập vận dụng (dễ → khó)

Các cell TODO **không được gọi khi chạy notebook** — hãy tự điền rồi tự chạy. Đáp án ở mục 19.

### Bài 1 — Đoán output

Đoán kết quả `bai1()` (không chạy).

In [26]:
def bai1():
    nums, k = [6, 6, 3, 8, 3, 8], 3
    s = {x for x in nums if x > k}
    return sorted(s), len(s), min(nums) < k

# Hãy đoán rồi mới chạy:  print(bai1())

### Bài 2 — Tính tay

Với `nums = [7, 3, 7, 9, 3, 5]` và `k = 3`: (a) kiểm tra `h = 6` có hợp lệ không; (b) tính số phép toán ít nhất. Điền hai biến dưới đây (không chạy hàm để lấy đáp án).

In [27]:
# TODO: tự tính tay rồi điền
bai2_h6_valid = None    # True/False
bai2_ans = None         # số phép toán ít nhất

### Bài 3 — Điền TODO

Viết lại lời giải một lượt duyệt: hàm `bai3(nums, k)` dùng `set` và `min` (không dùng set comprehension).

In [28]:
def bai3(nums, k):
    s = set()
    mi = float("inf")
    for x in nums:
        # TODO: nếu x < k thì trả -1
        ...
        # TODO: cập nhật mi và thêm x vào s
        ...
    # TODO: trả len(s) trừ 1 nếu mi == k
    ...

### Bài 4 — Tự viết lại

Viết `bai4(nums, k)` trả về **danh sách các `h`** được chọn theo thứ tự (chọn `h` lớn nhì rồi nhỏ dần, `k` ở cuối); nếu bất khả thi trả `None`. Ví dụ `bai4([5, 2, 5, 4, 5], 2) = [4, 2]`, `bai4([9, 7, 5, 3], 1) = [7, 5, 3, 1]`, `bai4([2, 1, 2], 2) = None`, `bai4([2, 2], 2) = []`.

In [29]:
def bai4(nums, k):
    # TODO: tự viết
    ...

### Bài 5 — Biến thể / mini challenge

Giả sử đề **không yêu cầu bằng `k`** mà chỉ yêu cầu **mọi phần tử bằng nhau (giá trị bất kỳ)**. Viết `bai5(nums)` trả về cặp `(số phép toán ít nhất, giá trị cuối cùng của mọi phần tử)`. Ví dụ `bai5([5, 2, 5, 4, 5]) = (2, 2)`, `bai5([7]) = (0, 7)`. (Gợi ý: giá trị cuối không thể nhỏ hơn `min(nums)`, và muốn ít phép toán nhất thì chọn nó bằng `min(nums)`.)

In [30]:
def bai5(nums):
    # TODO: tự viết
    ...

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.**
- Mức 1: `{x for x in nums if x > 3}` gồm những số nào lớn hơn `3`?
- Mức 2: các số `> 3` là `6, 6, 8, 8` ⇒ tập `{6, 8}`.
- Mức 3: kết quả `([6, 8], 2, False)` vì `min(nums) = 3` không nhỏ hơn `k = 3`.

**Bài 2.**
- Mức 1: các phần tử `> 6` là `7, 7, 9`; chúng có bằng nhau không?
- Mức 2: có `7` và `9` ⇒ `h = 6` **không** hợp lệ. Giá trị phân biệt `> 3` là `{7, 9, 5}`.
- Mức 3: đáp án `3` (và không có phần tử `< 3`).

**Bài 3.**
- Mức 1: giống Cách 3 trong mục 7.
- Mức 2: trong vòng lặp: `if x < k: return -1`, rồi `mi = min(mi, x)`, `s.add(x)`.
- Mức 3: `return len(s) - (1 if mi == k else 0)`.

**Bài 4.**
- Mức 1: lấy các giá trị phân biệt `> k`, sắp xếp **giảm dần**.
- Mức 2: các `h` là mỗi giá trị trong danh sách **trừ giá trị lớn nhất**, rồi `k` ở cuối, ví dụ `[5, 4]` ⇒ `h = 4, 2`.
- Mức 3: `vals = sorted({x for x in nums if x > k}, reverse=True)`; trả `vals[1:] + [k]` nếu `vals` không rỗng, `[]` nếu rỗng; `None` nếu `min(nums) < k`.

**Bài 5.**
- Mức 1: giá trị cuối cùng không thể nhỏ hơn `min(nums)`.
- Mức 2: chọn `k = min(nums)` thì `k` có sẵn trong mảng nên ta trừ `1`.
- Mức 3: trả `(len(set(nums)) - 1, min(nums))`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án đều được kiểm tra bằng `assert` (bài 4 và 5 còn đối chiếu bằng mô phỏng/BFS).

In [31]:
# --- Bài 1 ---
assert bai1() == ([6, 8], 2, False)
print("Bài 1:", bai1())

# --- Bài 2 ---
bai2_h6_valid_da = False
bai2_ans_da = 3
assert is_valid(6, [7, 3, 7, 9, 3, 5]) == bai2_h6_valid_da
assert Solution().minOperations([7, 3, 7, 9, 3, 5], 3) == bai2_ans_da
print("Bài 2:", bai2_h6_valid_da, bai2_ans_da)

# --- Bài 3 ---
def bai3_da(nums, k):
    s = set()
    mi = float("inf")
    for x in nums:
        if x < k:
            return -1
        mi = min(mi, x)
        s.add(x)
    return len(s) - (1 if mi == k else 0)

for _ in range(200):
    n = random.randint(1, 10)
    nums = [random.randint(1, 8) for _ in range(n)]
    k = random.randint(1, 8)
    assert bai3_da(nums, k) == Solution().minOperations(nums, k)
print("Bài 3 OK")

# --- Bài 4 ---
def bai4_da(nums, k):
    if min(nums) < k:
        return None
    vals = sorted({x for x in nums if x > k}, reverse=True)
    return vals[1:] + [k] if vals else []

assert bai4_da([5, 2, 5, 4, 5], 2) == [4, 2]
assert bai4_da([9, 7, 5, 3], 1) == [7, 5, 3, 1]
assert bai4_da([2, 1, 2], 2) is None
assert bai4_da([2, 2], 2) == []

def apply_hs(nums, hs, k):
    """Áp dụng lần lượt các h; kiểm tra từng h hợp lệ; cuối cùng mọi phần tử == k."""
    cur = list(nums)
    for h in hs:
        assert is_valid(h, cur), (h, cur)
        cur = [h if x > h else x for x in cur]
    return all(x == k for x in cur)

random.seed(3376)
for _ in range(300):
    n = random.randint(1, 8)
    nums = [random.randint(1, 8) for _ in range(n)]
    k = random.randint(1, 8)
    hs = bai4_da(nums, k)
    if hs is None:
        assert Solution().minOperations(nums, k) == -1
    else:
        assert len(hs) == Solution().minOperations(nums, k)
        assert apply_hs(nums, hs, k)
print("Bài 4 OK")

# --- Bài 5 ---
def bai5_da(nums):
    return (len(set(nums)) - 1, min(nums))

assert bai5_da([5, 2, 5, 4, 5]) == (2, 2)
assert bai5_da([7]) == (0, 7)
for _ in range(200):
    n = random.randint(1, 5)
    nums = [random.randint(1, 6) for _ in range(n)]
    ops, final = bai5_da(nums)
    # vét cạn BFS: thử mọi giá trị cuối v và lấy phương án ít phép toán nhất
    best = min((bfs_oracle(nums, v) for v in range(1, 7) if bfs_oracle(nums, v) != -1), default=None)
    assert best == ops, (nums, best, ops)
    assert bfs_oracle(nums, final) == ops
print("Bài 5 OK")

Bài 1: ([6, 8], 2, False)
Bài 2: False 3
Bài 3 OK
Bài 4 OK


Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (một lượt `set` + `min`):**

```python
class Solution:
    def minOperations(self, nums, k):
        s = set()
        mi = float("inf")
        for x in nums:
            if x < k:
                return -1
            mi = min(mi, x)
            s.add(x)
        return len(s) - int(mi == k)
```

**Công thức cần nhớ:** có `x < k` ⇒ `-1`; ngược lại ⇒ **số giá trị phân biệt lớn hơn `k`**.

**Checklist trước khi submit:**

- [ ] Tên method đúng: `minOperations(self, nums, k)`.
- [ ] Kiểm tra `x < k` (nghiêm ngặt) ⇒ `-1`; phần tử bằng `k` vẫn hợp lệ.
- [ ] Đếm giá trị **phân biệt**, không đếm phần tử.
- [ ] Chỉ trừ `1` khi `k` thực sự có mặt (`mi == k`).
- [ ] Không sửa mảng đầu vào khi mô phỏng (dùng bản sao).
- [ ] Đã thử: `k` có sẵn, `k` không có sẵn, mọi phần tử bằng nhau, `n = 1`, có phần tử `< k`.